# Bike Demo
adapted from https://github.com/brightway-lca/from-the-ground-up/tree/main/try.brightway.dev

In [1]:
# imports tell us which libraries we need to run the code.
import bw2calc as bc
import bw2data as bd
import bw2io as bi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Projects and Database setup

The first thing to learn about `bw2data` is the concept of projects. Each project is self-contained, and independent of other projects. Each has its own subdirectory. This can lead to data duplication, but helps keep each project safe from the changes in the others.

We start in the `default` project:

In [2]:
bd.projects.current

'default'

Let's create a new project:

In [3]:
# deleting the project so I can rerun and check everything works but will remove this before students use it
bd.projects.delete_project(name='demo', delete_dir=True)

'default'

In [5]:
bd.projects.set_current(name='demo')
bd.projects.current

'demo'

To enter our data into BW, we need to create the nodes (or activities), and then the edges (or exchanges). We will create these nodes in a `Database`. A database in BW is just a collection of nodes - it can be large or small, there aren't any general rules.

Let's check what databases we have:

In [6]:
bd.databases

Databases dictionary with 0 objects

This should be empty, we haven't created any yet.

Let's register a new database:

In [7]:
bike_db = bd.Database("bike")
bike_db.register()
bd.databases # now when we check this, we see our new database

Databases dictionary with 1 object(s):
	bike

We wont be using ecoinvent in this example, but we can access Bioshpere3 (which is a database full of elementary flow data and is open access) and we can access the LCIA methods so let's add those (this might take a minute)

In [8]:
bi.create_default_biosphere3(overwrite=False)
bi.create_default_lcia_methods(overwrite=False, rationalize_method_names=False, shortcut=True)

Writing activities to SQLite3 database:


Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.01 seconds


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 04/20/2026 14:08:05
  Finished: 04/20/2026 14:08:05
  Total time elapsed: 00:00:00
  CPU %: 7.40
  Memory %: 1.31
Created database: biosphere3
Wrote 762 LCIA methods with 227223 characterization factors


Now when we look at our databases again, we can see that bioshpere3 is included

In [13]:
bd.databases

Databases dictionary with 2 object(s):
	bike
	biosphere3

We should make it easy to access for later

In [14]:
bs_db = bd.Database("biosphere3")

## 2. Activities and Exchanges

Next we can start to create our foreground, in this case, a bike.

We will start with creating our nodes or activities:

In [ ]:
bike = bike_db.new_activity(
    name ='bike',
    unit ='unit',
    location = 'DK',
    type = 'product',
    code = 'bike', # codes must be unique within a database, but can be anything. See what happens if you try to save this activity again.
)
bike.save()

IntegrityError: UNIQUE constraint failed: activitydataset.database, activitydataset.code

In [10]:
bike_production = bike_db.new_activity(
    name ='bike production',
    location ='DK',
    type = 'process',
    code = 'bike_production',
)
bike_production.save()

Now try to create the activities for carbon fibre and carbon fibre production

In [ ]:
# note to remove the values for these so the students have to write them
cf = bike_db.new_activity(
    name = 'carbon fibre', 
    unit = 'kg',
    location = 'DE',
    type = 'product',
    code = 'cf',
)
cf.save()

cf_production = bike_db.new_activity(
    name = 'carbon fibre production',
    location = 'DE',
    type = 'process',
    code = 'cf_production',
)
cf_production.save()

In our process graph, we can see that some CO2 is also created during this step. We will include this later. For now, let's make the natural gas activities

In [12]:
# Try to add your own activities here! 

Now we need to connect our activities together using exchanges! 

We'll start from the bike again:

In [18]:
# This is the production exchange, which links the production of the bike to the bike activity. The amount is usually 1.
bike_production.new_exchange(
    input = bike,
    type = 'production',
    amount = 1,
).save()

# Consumption exchanges represent how much carbon fibre is used per bike production. 
bike_production.new_exchange(
    input = cf,
    type = 'consumption',
    amount = 2.5,
).save()

# Again we will need a production exchange for the carbon fibre production.
cf_production.new_exchange(
    input = cf,
    type = 'production',
    amount = 1,
).save()

Now try to continue building the exchanges based on the natural gas you have created before.
Ignore the CO2 again for now, we will add that later.

In [15]:
# Try to make the natural gas exchanges here!

Great job! Now we can have a little look at what we've made:

In [ ]:
# Let's check our activities
for act in bike_db:
    print(act)

'carbon fibre production' (None, DE, None)
'carbon fibre' (kg, GLO, None)
'bike' (number of bikes, GLO, None)
'bike production' (None, DK, None)


In [ ]:
# Now let's check out the exchanges for our carbon fibre
# You can do this by just printing the exchanges
for exc in cf_production.exchanges():
    print(exc)

Exchange: 1 kg 'carbon fibre' (kg, GLO, None) to 'carbon fibre production' (None, DE, None)>


In [21]:
# Or you can use Python's built in list comprehension
[exc for exc in cf_production.exchanges()]

[Exchange: 1 kg 'carbon fibre' (kg, GLO, None) to 'carbon fibre production' (None, DE, None)>]

Oh yeah, it's missing our CO2. We should add that next

## 3. Working with larger datasets

First we need to find the CO2 that we want to use in the bioshpere3 database. Let's look for it using list comprehension